# Clip animate su TPU: Wan 2.2 grande (A14B) con MaxDiffusion

Alternativa a ComfyUI: gira sulla **TPU gratuita di Kaggle** (quota separata da quella GPU)
con [MaxDiffusion](https://github.com/AI-Hypercomputer/maxdiffusion), la libreria di Google in JAX.
Usa **Wan 2.2 I2V A14B** (il modello grande, ~27 miliardi di parametri in due "esperti")
a **1280x720**: molto meglio del 5B a 832x480 che entra nella T4.

Il notebook anima **un'immagine per scena** con il suo prompt di movimento,
carica il modello **una volta sola** e salva le clip in `/kaggle/working/clip/<id>_tpu.mp4`.

## Prima di partire (una volta)
1. Sul PC: `python prepara_storia.py storie/DaniTeo --tpu` → crea `storie/DaniTeo/kaggle_tpu/`
   (le immagini scelte + `clip.txt`).
2. Su Kaggle: **Create → New Dataset**, trascinate dentro *il contenuto* di `kaggle_tpu/`, nome per es. `daniteo-scene`.
3. In questo notebook, pannello a destra:
   - **Accelerator → TPU v5e-8** (o la TPU disponibile);
   - **Internet → On** (scarica MaxDiffusion e il modello da Hugging Face);
   - **Add Input** → il dataset del punto 2.

## Come usarlo
- **Prova (consigliata la prima volta):** lasciate `PROVA = True` e fate **Run All**:
  genera solo la prima scena e stampa i tempi. Guardate la clip in `/kaggle/working/clip/`.
- **Tutte le scene:** `PROVA = False`, poi **Save Version → Save & Run All (Commit)**: gira in background
  anche a browser chiuso (massimo ~9 ore a sessione). Alla fine, dalla pagina della versione, scheda
  **Output**, scaricate `clip_tpu.zip`.
- Sul PC: estraete lo zip in `output/DaniTeo_video/` e rimontate:
  `python monta_video.py storie/DaniTeo --sottotitoli-nel-video` (prende da sé le clip più recenti).

## Note oneste
- È sperimentale: MaxDiffusion è pensato per le TPU di Google Cloud (v6e); sulla v5e di Kaggle dovrebbe
  andare ma non è documentato. La cella 2 controlla RAM e disco prima di scaricare ~80 GB di modello.
- La **prima** clip è lenta (download del modello + compilazione XLA); le successive molto di più veloci.
- Se va in errore di memoria: `RISOLUZIONE = (832, 480)` oppure `MODELLO = 'wan21'` (Wan 2.1 I2V 14B, più leggero).

In [ ]:
# ===== Impostazioni =====
PROVA = True                 # True: solo la prima scena (per vedere tempi e qualità). False: tutte
SOLO = []                    # es. ['14_proposta', '17_primo_ballo'] per rifare solo quelle
MODELLO = 'wan22'            # 'wan22' = Wan 2.2 I2V A14B (migliore) | 'wan21' = Wan 2.1 I2V 14B (più leggero)
RISOLUZIONE = (1280, 720)    # (larghezza, altezza). Più veloce/leggero: (832, 480)
FOTOGRAMMI = 81              # 81 a 16 fps = 5 secondi. Deve essere 4k+1 (49, 65, 81...)
PASSI = 40                   # passi di denoising (40 = qualità piena; 30 = più veloce)
ACCELERA = None              # None | 'sen' (SenCache, ~1.4x) | 'mag' (MagCache, ~1.8x, un filo meno fedele)
SEED = 42
NEGATIVO = ('static image, frozen, no motion, blurry, low quality, distorted face, deformed face, '
            'morphing, melting, extra fingers, deformed hands, extra limbs, duplicate person, '
            'flickering, jitter, overexposed, text, subtitles, watermark, logo')

In [ ]:
# ===== 1. Controllo della macchina =====
import glob, os, shutil, subprocess

tpu = glob.glob('/dev/accel*') + glob.glob('/dev/vfio/*')
print('TPU:', f'{len(tpu)} dispositivi' if tpu else 'NON TROVATA -> Settings > Accelerator > TPU')
ram = os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / 2**30
print(f'RAM: {ram:.0f} GB')
for cartella in ('/tmp', '/kaggle/working'):
    libero = shutil.disk_usage(cartella).free / 2**30
    print(f'Disco libero in {cartella}: {libero:.0f} GB')
if shutil.disk_usage('/tmp').free / 2**30 < 100:
    print('ATTENZIONE: il modello Wan 2.2 A14B occupa ~80 GB: potrebbe non starci. '
          "Provate MODELLO = 'wan21'.")
if ram < 120:
    print('ATTENZIONE: poca RAM per caricare il modello grande: se il kernel muore, '
          "usate MODELLO = 'wan21'.")
subprocess.run(['python3', '--version'])

In [ ]:
# ===== 2. Installazione di MaxDiffusion (5-10 minuti) =====
# MaxDiffusion vuole Python 3.12: lo mettiamo in un ambiente separato con uv,
# senza toccare il Python del notebook.
import os, subprocess

REPO = '/tmp/maxdiffusion'
VENV = '/tmp/md_venv'
PY = f'{VENV}/bin/python'

def esegui(comando, cwd=None):
    print('$', comando, flush=True)
    subprocess.run(comando, shell=True, check=True, cwd=cwd, executable='/bin/bash')

if not os.path.exists(f'{VENV}/.pronto'):
    esegui('pip install -q uv')
    if not os.path.isdir(REPO):
        esegui(f'git clone -q --depth 1 https://github.com/AI-Hypercomputer/maxdiffusion.git {REPO}')
    esegui(f'python3 -m uv venv --python 3.12 --seed {VENV}')
    uv = f'python3 -m uv pip install -q --python {PY}'
    # Gli stessi passi di setup.sh (MODE=stable DEVICE=tpu), senza la parte apt/gcsfuse.
    esegui(f'{uv} -U --resolution=lowest '
           '-r dependencies/requirements/generated_requirements/requirements.txt '
           '-r src/install_maxdiffusion_extra_deps/extra_deps_from_github.txt', cwd=REPO)
    esegui(f"{uv} 'jax[tpu]>0.4' -f https://storage.googleapis.com/jax-releases/libtpu_releases.html")
    esegui(f'{uv} --no-deps .', cwd=REPO)
    open(f'{VENV}/.pronto', 'w').close()

esegui(f'{PY} -c "import jax; print(jax.devices())"')

In [ ]:
# ===== 3. Scene da animare (dal dataset caricato) =====
import glob, json, os
from PIL import Image

trovati = glob.glob('/kaggle/input/**/clip.txt', recursive=True)
assert trovati, ('Non trovo clip.txt: aggiungete il dataset fatto con '
                 '"python prepara_storia.py storie/DaniTeo --tpu" (Add Input).')
DATASET = os.path.dirname(trovati[0])
print('Dataset:', DATASET)

L, A = RISOLUZIONE
os.makedirs('/tmp/scene_in', exist_ok=True)
lavori = []
for riga in open(trovati[0], encoding='utf-8'):
    if '|' not in riga:
        continue
    scena, prompt = (x.strip() for x in riga.split('|', 1))
    if SOLO and scena not in SOLO:
        continue
    immagini = [f for f in glob.glob(f'{DATASET}/{scena}.*') if not f.endswith('.txt')]
    if not immagini:
        print(f'  {scena}: manca l\'immagine, salto')
        continue
    # Ritaglio al centro nel formato giusto e ridimensiono: il modello la riceve già pronta.
    img = Image.open(immagini[0]).convert('RGB')
    scala = max(L / img.width, A / img.height)
    img = img.resize((round(img.width * scala), round(img.height * scala)), Image.LANCZOS)
    x, y = (img.width - L) // 2, (img.height - A) // 2
    percorso = f'/tmp/scene_in/{scena}.png'
    img.crop((x, y, x + L, y + A)).save(percorso)
    lavori.append({'id': scena, 'prompt': prompt, 'immagine': percorso})

if PROVA:
    lavori = lavori[:1]
json.dump({'negativo': NEGATIVO, 'lavori': lavori}, open('/tmp/lavori.json', 'w'))
print(f'{len(lavori)} scene da fare:', ', '.join(l['id'] for l in lavori))

In [ ]:
%%writefile /tmp/wan_tpu_clip.py
"""Genera più clip con generate_wan.py di MaxDiffusion caricando il modello UNA volta sola.

generate_wan.py carica il modello e poi chiama _warmup_and_generate() con un solo prompt
e una sola immagine. Qui la sostituiamo con un ciclo sulle scene: per ognuna cambiamo
prompt / immagine / nome nella configurazione e richiamiamo l'originale (dalla seconda
scena la compilazione è già fatta).

Uso: python wan_tpu_clip.py lavori.json cartella_uscita config.yml chiave=valore ...
"""
import json
import shutil
import sys
import time
import traceback
from pathlib import Path

dati = json.loads(Path(sys.argv[1]).read_text())
uscita = Path(sys.argv[2])
uscita.mkdir(parents=True, exist_ok=True)
sys.argv = ['generate_wan.py', *sys.argv[3:]]

from absl import app
import maxdiffusion.generate_wan as gw

originale = gw._warmup_and_generate


def tutte_le_scene(config, pipeline, filename_prefix, writer, load_time, aot_cache_dir):
    chiavi = config.get_keys()  # la config non si modifica con setattr: si cambia il dizionario
    chiavi['negative_prompt'] = dati['negativo']
    lavori = dati['lavori']
    fatte = []
    for n, lavoro in enumerate(lavori, 1):
        destinazione = uscita / f"{lavoro['id']}_tpu.mp4"
        if destinazione.exists():
            print(f"[{n}/{len(lavori)}] {lavoro['id']}: già fatta", flush=True)
            continue
        chiavi['prompt'] = lavoro['prompt']
        chiavi['image_url'] = lavoro['immagine']
        chiavi['run_name'] = lavoro['id']
        inizio = time.time()
        try:
            salvati = originale(config, pipeline, filename_prefix, writer, load_time, aot_cache_dir)
        except Exception:
            traceback.print_exc()
            print(f"[{n}/{len(lavori)}] {lavoro['id']}: ERRORE, passo alla prossima", flush=True)
            continue
        if salvati:
            shutil.move(salvati[0], destinazione)
            fatte.append(str(destinazione))
            print(f"[{n}/{len(lavori)}] {lavoro['id']}: fatta in {time.time() - inizio:.0f} s "
                  f"-> {destinazione}", flush=True)
    return fatte


gw._warmup_and_generate = tutte_le_scene

if __name__ == '__main__':
    with gw.transformer_engine_context():
        app.run(gw.main)

In [ ]:
# ===== 4. Generazione =====
import os, subprocess, time

CONFIG = {'wan22': 'base_wan_i2v_27b.yml', 'wan21': 'base_wan_i2v_14b.yml'}[MODELLO]
L, A = RISOLUZIONE
n_chip = 8  # v5e-8: batch 1 in tutto = 1/8 per chip
argomenti = [
    f'{REPO}/src/maxdiffusion/configs/{CONFIG}',
    'run_name=cartone', 'output_dir=/tmp/clip_grezze', 'write_metrics=False',
    f'width={L}', f'height={A}', f'num_frames={FOTOGRAMMI}', f'num_inference_steps={PASSI}',
    f'seed={SEED}', 'fps=16', 'attention=flash',
    f'per_device_batch_size={1 / n_chip}',
    'ici_data_parallelism=1', 'ici_fsdp_parallelism=1', 'ici_context_parallelism=-1',
    'ici_tensor_parallelism=1',
    'weights_dtype=bfloat16', 'activations_dtype=bfloat16',
    'jax_cache_dir=/tmp/jax_cache',
]
if ACCELERA == 'sen':
    argomenti.append('use_sen_cache=True')
elif ACCELERA == 'mag':
    argomenti.append('use_magcache=True')

ambiente = {**os.environ,
            'HF_HUB_CACHE': '/tmp/hf_cache',   # fuori da /kaggle/working (limite 20 GB)
            'PYTHONUNBUFFERED': '1',
            'LIBTPU_INIT_ARGS': ('--xla_tpu_enable_async_collective_fusion=true '
                                 '--xla_tpu_enable_async_collective_fusion_fuse_all_reduce=true '
                                 '--xla_tpu_enable_async_collective_fusion_multiple_steps=true '
                                 '--xla_tpu_overlap_compute_collective_tc=true '
                                 '--xla_enable_async_all_reduce=true')}

inizio = time.time()
with open('/kaggle/working/generazione.log', 'w') as log:
    p = subprocess.Popen([PY, '/tmp/wan_tpu_clip.py', '/tmp/lavori.json', '/kaggle/working/clip',
                          *argomenti], cwd=REPO, env=ambiente, text=True,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    for riga in p.stdout:
        log.write(riga); log.flush()
        # Solo le righe utili; il log completo è in /kaggle/working/generazione.log
        if any(k in riga for k in ('[', 'load_time', 'compile_time', 'generation_time', 'Error',
                                   'error', 'Traceback', 'TIMING', 'Load', 'Compile', 'Inference',
                                   'Downloading', 'Fetching')):
            print(riga, end='', flush=True)
    p.wait()
print(f'\nFinito (codice {p.returncode}) in {(time.time() - inizio) / 60:.0f} minuti')

In [ ]:
# ===== 5. Risultato: zip da scaricare =====
import glob, os, shutil
clip = sorted(glob.glob('/kaggle/working/clip/*.mp4'))
for c in clip:
    print(f'{os.path.getsize(c) / 2**20:6.1f} MB  {os.path.basename(c)}')
if clip:
    shutil.make_archive('/kaggle/working/clip_tpu', 'zip', '/kaggle/working/clip')
    print('\nScaricate /kaggle/working/clip_tpu.zip (scheda Output) ed estraetelo in output/DaniTeo_video/')
else:
    print('Nessuna clip: guardate /kaggle/working/generazione.log')

In [ ]:
# (Facoltativo) Guardare la prima clip qui nel notebook
from IPython.display import Video
import glob
clip = sorted(glob.glob('/kaggle/working/clip/*.mp4'))
Video(clip[0], embed=True, width=640) if clip else print('nessuna clip')